# 📊 Phase 1 — Part 7: Evaluate ALL Models & Create ASR Module

---

## What this notebook does
1. **Evaluates** every trained model on the TEST set (never seen during training)
2. **Compares** WER scores across all models and dialects
3. **Identifies** the BEST model
4. **Copies** the best model to `BEST_ASR_MODEL/`
5. **Creates** `asr_module.py` — the handoff file Phase 2 will import

## What gets saved
| File | Location | Purpose |
|------|----------|--------|
| WER results JSON | `results/all_wer_scores.json` | All scores in one file |
| Best model copy | `models/BEST_ASR_MODEL/` | Used by Phase 2 |
| ASR module script | `scripts/asr_module.py` | Phase 2 imports this |
| Evaluation script | `scripts/evaluate_all.py` | Code backup |

---

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os
config = json.load(open('/content/drive/MyDrive/NSU_499A/config.json'))
BASE = config['base_path']
print(f'✅ Project root: {BASE}')

In [ ]:
!pip install -q transformers datasets evaluate jiwer accelerate
print('✅ Packages installed')

## Step 1: Discover All Trained Models
This cell scans your `models/` directory and finds every model that has been trained.

In [ ]:
models_dir = os.path.join(BASE, 'models')

# Find all model folders that contain a config.json (= trained model)
trained_models = []
for name in sorted(os.listdir(models_dir)):
    model_path = os.path.join(models_dir, name)
    if os.path.isdir(model_path):
        has_config = os.path.exists(os.path.join(model_path, 'config.json'))
        has_weights = (
            os.path.exists(os.path.join(model_path, 'model.safetensors')) or
            os.path.exists(os.path.join(model_path, 'pytorch_model.bin'))
        )
        if has_config and has_weights:
            # Determine dialect from folder name
            if 'dhaka' in name:
                dialect = 'dhaka'
            elif 'chittagong' in name:
                dialect = 'chittagong'
            else:
                continue
            
            # Determine model type
            if 'whisper' in name:
                mtype = 'whisper'
            else:
                mtype = 'ctc'
            
            trained_models.append({
                'name': name,
                'path': model_path,
                'dialect': dialect,
                'type': mtype
            })

print(f'🔍 Found {len(trained_models)} trained models:\n')
for m in trained_models:
    print(f'  ✅ {m["name"]} ({m["type"]}, {m["dialect"]})')

if len(trained_models) == 0:
    print('\n❌ No trained models found! Complete Parts 5 and 6 first.')

## Step 2: Evaluate Every Model on Test Set

### What happens:
- For each model, loads the test CSV for its dialect
- Runs every test clip through the model
- Compares predicted text to ground truth
- Computes WER (Word Error Rate)

### ⏱️ This takes ~5-15 minutes depending on number of models

In [ ]:
from transformers import pipeline
import pandas as pd
import evaluate

wer_metric = evaluate.load('wer')
all_results = {}

for m in trained_models:
    name = m['name']
    model_path = m['path']
    dialect = m['dialect']
    mtype = m['type']
    
    print(f'\n📊 Evaluating: {name}')
    
    # Load test data
    test_csv = os.path.join(BASE, f'dataset/{dialect}_test.csv')
    if not os.path.exists(test_csv):
        print(f'  ❌ Test CSV not found: {test_csv}')
        continue
    
    test_df = pd.read_csv(test_csv)
    print(f'  Test samples: {len(test_df)}')
    
    # Load model as ASR pipeline
    try:
        if mtype == 'whisper':
            asr = pipeline('automatic-speech-recognition',
                          model=model_path, device=0)
        else:
            asr = pipeline('automatic-speech-recognition',
                          model=model_path, device=0)
    except Exception as e:
        print(f'  ❌ Failed to load: {e}')
        continue
    
    # Run predictions
    preds, refs = [], []
    for idx, row in test_df.iterrows():
        try:
            result = asr(row['file_path'])
            preds.append(result['text'])
            refs.append(str(row['transcript']))
        except Exception as e:
            continue
        
        if (idx + 1) % 50 == 0:
            print(f'  Progress: {idx+1}/{len(test_df)}')
    
    # Compute WER
    if len(preds) > 0:
        wer = wer_metric.compute(predictions=preds, references=refs)
        wer_pct = round(wer * 100, 2)
        all_results[name] = wer_pct
        print(f'  ✅ WER = {wer_pct}%')
    else:
        print(f'  ❌ No predictions made')
    
    # Free GPU memory
    del asr
    import torch
    torch.cuda.empty_cache()

print(f'\n{"="*60}')
print('EVALUATION COMPLETE')
print(f'{"="*60}')

## Step 3: Results Summary Table

In [ ]:
# ════════════════════════════════════════════════════════════════
# RESULTS TABLE
# ════════════════════════════════════════════════════════════════

if len(all_results) > 0:
    print(f'\n{"Model":<35} {"WER (%)":<10} {"Rank"}')
    print('─' * 55)
    
    sorted_results = sorted(all_results.items(), key=lambda x: x[1])
    best_model_name = sorted_results[0][0]
    
    for rank, (name, wer) in enumerate(sorted_results, 1):
        marker = ' ← BEST' if rank == 1 else ''
        print(f'  {name:<33} {wer:<10} #{rank}{marker}')
    
    print(f'\n🏆 Best model: {best_model_name} (WER = {all_results[best_model_name]}%)')
else:
    print('❌ No results to show. Evaluate models first.')
    best_model_name = None

## Step 4: Save Results to Google Drive

In [ ]:
# Save WER scores
results_path = os.path.join(BASE, 'results/all_wer_scores.json')
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'✅ Results saved: {results_path}')

## Step 5: Copy Best Model to `BEST_ASR_MODEL/`

This copies the winning model to a standard location that Phase 2 will use.

In [ ]:
import shutil

if best_model_name:
    src = os.path.join(BASE, f'models/{best_model_name}')
    dst = os.path.join(BASE, 'models/BEST_ASR_MODEL')
    
    # Clear destination
    if os.path.exists(dst):
        shutil.rmtree(dst)
    
    # Copy all files (not checkpoint subdirs)
    os.makedirs(dst, exist_ok=True)
    for item in os.listdir(src):
        s = os.path.join(src, item)
        d = os.path.join(dst, item)
        if os.path.isfile(s):
            shutil.copy2(s, d)
            print(f'  📄 Copied: {item}')
    
    # Save metadata
    meta = {
        'source_model': best_model_name,
        'wer': all_results[best_model_name],
        'all_results': all_results
    }
    with open(os.path.join(dst, 'best_model_info.json'), 'w') as f:
        json.dump(meta, f, indent=2)
    
    print(f'\n✅ Best model copied to: models/BEST_ASR_MODEL/')
    print(f'   Source: {best_model_name}')
    print(f'   WER: {all_results[best_model_name]}%')
else:
    print('❌ No best model identified. Run evaluation first.')

## Step 6: Create `asr_module.py` — Phase 1 → Phase 2 Bridge

This is the file that **Phase 2** imports to use your best ASR model.

### How Phase 2 will use it:
```python
from asr_module import BanglaASR
asr = BanglaASR()
result = asr.transcribe('patient_recording.wav')
transcript = result['transcript']  # → goes into NER
```

In [ ]:
asr_module_code = '''# asr_module.py — Phase 1 → Phase 2 Bridge
# This file is imported by Phase 2 to get Bangla transcripts from audio.

from transformers import pipeline
import os

class BanglaASR:
    def __init__(self, model_path='models/BEST_ASR_MODEL'):
        """
        Initialize the Bangla ASR model.
        
        Args:
            model_path: Path to the fine-tuned model directory.
                        Default uses the best model from Phase 1.
        """
        self.model_path = model_path
        self.asr = pipeline(
            'automatic-speech-recognition',
            model=model_path,
            device=0 if __import__('torch').cuda.is_available() else -1
        )
        print(f'BanglaASR loaded from: {model_path}')
    
    def transcribe(self, audio_path: str) -> dict:
        """
        Transcribe a Bangla audio file to text.
        
        Args:
            audio_path: Path to a WAV file (16kHz mono recommended)
        
        Returns:
            dict with keys: transcript, audio_file, model, language
        """
        result = self.asr(audio_path)
        return {
            'transcript': result['text'],
            'audio_file': os.path.basename(audio_path),
            'model': 'bangla-asr-finetuned',
            'language': 'bn'
        }


# Quick test when run directly
if __name__ == '__main__':
    import sys
    asr = BanglaASR()
    if len(sys.argv) > 1:
        result = asr.transcribe(sys.argv[1])
        print(f"Transcript: {result['transcript']}")
    else:
        print('Usage: python asr_module.py <audio_file.wav>')
'''

module_path = os.path.join(BASE, 'scripts/asr_module.py')
with open(module_path, 'w') as f:
    f.write(asr_module_code)
print(f'✅ ASR module saved: {module_path}')
print('   Phase 2 will import this file to use your best ASR model.')

## Step 7: Save Evaluation Script

In [ ]:
eval_script = '''# evaluate_all.py — Evaluate all trained ASR models
# See Phase1_Part7_Evaluate.ipynb for full interactive version

from transformers import pipeline
import pandas as pd, evaluate, json

# Add your models here:
MODELS = {
    'whisper-small-dhaka':    ('openai', 'models/whisper-small-dhaka'),
    'wav2vec2-xlsr-dhaka':    ('ctc',    'models/wav2vec2-xlsr-dhaka'),
    'wavlm-large-dhaka':     ('ctc',    'models/wavlm-large-dhaka'),
}

wer_metric = evaluate.load('wer')
results = {}

for name, (mtype, mpath) in MODELS.items():
    dialect = name.split('-')[-1]
    test_df = pd.read_csv(f'dataset/{dialect}_test.csv')
    asr = pipeline('automatic-speech-recognition', model=mpath)
    preds = [asr(row['file_path'])['text'] for _, row in test_df.iterrows()]
    refs = test_df['transcript'].tolist()
    wer = wer_metric.compute(predictions=preds, references=refs)
    results[name] = round(wer * 100, 2)
    print(f'{name}: WER = {wer*100:.2f}%')

json.dump(results, open('results/all_wer_scores.json', 'w'), indent=2)
'''
with open(os.path.join(BASE, 'scripts/evaluate_all.py'), 'w') as f:
    f.write(eval_script)
print('✅ Script saved: scripts/evaluate_all.py')

---
## ✅ PHASE 1 COMPLETE!

### Summary of everything saved in Google Drive:

```
NSU_499A/
├── raw_audio/dhaka/             ← Downloaded audio
├── raw_audio/chittagong/        ← Downloaded audio  
├── clips/dhaka/                 ← 10-sec clips
├── clips/chittagong/            ← 10-sec clips
├── dataset/
│   ├── dhaka_raw.csv            ← Transcriptions (auto + manual)
│   ├── dhaka_train.csv          ← Training split
│   ├── dhaka_val.csv            ← Validation split
│   ├── dhaka_test.csv           ← Test split
│   └── (same for chittagong)
├── models/
│   ├── whisper-small-dhaka/     ← Trained Whisper
│   ├── wav2vec2-xlsr-dhaka/     ← Trained Wav2Vec2
│   ├── wavlm-large-dhaka/       ← Trained WavLM
│   ├── BEST_ASR_MODEL/          ← ⭐ Best model for Phase 2
│   └── (same for chittagong)
├── scripts/
│   ├── cut_audio.py
│   ├── auto_transcribe.py
│   ├── split_dataset.py
│   ├── train_whisper.py
│   ├── train_wav2vec2.py
│   ├── evaluate_all.py
│   └── asr_module.py            ← ⭐ Phase 2 imports this
├── results/
│   └── all_wer_scores.json      ← All WER scores
└── config.json
```

### How to resume if anything was lost:
1. Mount Google Drive → everything is there
2. Install packages (they don't persist)
3. Load config.json
4. Continue from where you left off

### For Phase 2:
Phase 2 will use `scripts/asr_module.py` and `models/BEST_ASR_MODEL/`

---